# Subsample data (2500 fps to 250 fps)


In [3]:

import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import decimate

#Directories
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUBSAMPLE_FACTOR = 10  # 2500 fps → 250 fps

#Find smooth files
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv"))
if not csv_files:
    raise FileNotFoundError(f"No SMOOTH CSV files found in {INPUT_DIR}")

for path in csv_files:
    fname = os.path.basename(path)
    print("Decimating:", fname)

    df = pd.read_csv(path)
    n_out = len(df.iloc[::SUBSAMPLE_FACTOR])
    sub_df = pd.DataFrame()
    sub_df["frame"] = np.arange(n_out)

    for col in df.columns[1:]:  # skip frame
        sig = df[col].values
        out = np.full(n_out, np.nan)

        if np.all(np.isnan(sig)):
            out = sig[::SUBSAMPLE_FACTOR]
            sub_df[col] = out
            continue

        isnan = np.isnan(sig)
        i = 0
        while i < len(sig):
            if not isnan[i]:
                j = i
                while j < len(sig) and not isnan[j]:
                    j += 1
                seg = sig[i:j]

                # where does this segment land in the output array?
                out_i = int(np.ceil(i / SUBSAMPLE_FACTOR))
                out_j = int(np.ceil(j / SUBSAMPLE_FACTOR))

                if len(seg) >= SUBSAMPLE_FACTOR * 3:
                    dec = decimate(seg, SUBSAMPLE_FACTOR, ftype='fir', zero_phase=True)
                else:
                    #interpolation instead of skipping
                    x_old = np.arange(len(seg))
                    x_new = np.linspace(0, len(seg) - 1, int(np.ceil(len(seg) / SUBSAMPLE_FACTOR)))
                    dec = np.interp(x_new, x_old, seg)

                # trim to expected length
                expected = out_j - out_i
                dec = dec[:expected]

                out[out_i : out_i + len(dec)] = dec
                i = j
            else:
                i += 1

        sub_df[col] = out

    out_name = fname.replace("_SMOOTH.csv", "_SUB.csv")
    sub_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)
    print(f"  → Saved {out_name}  ({len(sub_df)} frames)")

print(f"\nDone. {len(csv_files)} file(s) decimated and saved to {OUTPUT_DIR}")

Decimating: Trial2_180rpmxyzpts_SMOOTH.csv
  → Saved Trial2_180rpmxyzpts_SUB.csv  (81 frames)
Decimating: Trial2_200rpmxyzpts_SMOOTH.csv
  → Saved Trial2_200rpmxyzpts_SUB.csv  (24 frames)
Decimating: Trial3_180rpmxyzpts_SMOOTH.csv
  → Saved Trial3_180rpmxyzpts_SUB.csv  (24 frames)
Decimating: Trial4_400rpmxyzpts_SMOOTH.csv
  → Saved Trial4_400rpmxyzpts_SUB.csv  (45 frames)
Decimating: Trial5_180rpmxyzpts_SMOOTH.csv
  → Saved Trial5_180rpmxyzpts_SUB.csv  (66 frames)
Decimating: Trial5_400rpmxyzpts_SMOOTH.csv
  → Saved Trial5_400rpmxyzpts_SUB.csv  (68 frames)
Decimating: Trial7_400rpmxyzpts_SMOOTH.csv
  → Saved Trial7_400rpmxyzpts_SUB.csv  (68 frames)

Done. 7 file(s) decimated and saved to ./../../dataFolders/MuscaChasingBeads/xyz_Subsampled


# Turning angle(center-based)

In [5]:
import os
import glob
import numpy as np
import pandas as pd

# ── Config ─────────────────────────────────────────────────────────────
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based"

os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS = 250
DT  = 1 / FPS
TRIM = 4   # ~16 ms

# ── Helpers ───────────────────────────────────────────────────────────
def normalize(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0] = 1
    return v / n

def turning_angle_3d(v_in, v_out):
    dot   = np.sum(v_in * v_out, axis=1)
    cross = np.linalg.norm(np.cross(v_in, v_out), axis=1)
    return np.degrees(np.arctan2(cross, dot))

def turning_angle_2d(v_in, v_out):
    dot   = np.sum(v_in * v_out, axis=1)
    cross = v_in[:, 0]*v_out[:, 1] - v_in[:, 1]*v_out[:, 0]
    return np.degrees(np.arctan2(cross, dot))

# ── Process ───────────────────────────────────────────────────────────
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SUB.csv"))

for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace("_SUB.csv", "")
    print(f"Processing: {fname}")

    df = pd.read_csv(path)

    center = df[["center_X","center_Y","center_Z"]].values
    frames_all = df["frame"].values

    # ── Apply SAME mask ───────────────────────────────────────────────
    valid_mask = ~np.all(np.isnan(center), axis=1)

    center = center[valid_mask]
    frames_all = frames_all[valid_mask]

    if len(center) < 3:
        print("  ⚠ Skipping (too few valid points)")
        continue

    # ── CENTERED VECTORS ─────────────────────────────────────────────
    v_in  = center[1:-1] - center[:-2]
    v_out = center[2:]   - center[1:-1]

    v_in  = normalize(v_in)
    v_out = normalize(v_out)

    # ── Turning angles ───────────────────────────────────────────────
    turn_3d = turning_angle_3d(v_in, v_out)

    turn_xy_raw = turning_angle_2d(
        normalize(v_in[:, :2]),
        normalize(v_out[:, :2])
    )

    turn_yz_raw = turning_angle_2d(
        normalize(v_in[:, 1:]),
        normalize(v_out[:, 1:])
    )

    # ── UNWRAP ───────────────────────────────────────────────────────
    turn_xy = np.degrees(np.unwrap(np.radians(turn_xy_raw)))
    turn_yz = np.degrees(np.unwrap(np.radians(turn_yz_raw)))

    # ── Frame + time alignment ───────────────────────────────────────
    frames = frames_all[1:-1]
    time   = frames / FPS

    # ── Safety check ─────────────────────────────────────────────────
    if not (len(frames) == len(turn_3d) == len(turn_xy)):
        print("  ⚠ Length mismatch — skipping")
        continue

    # ── SAVE FULL ───────────────────────────────────────────────────
    out_df = pd.DataFrame({
        "frame": frames,
        "time_sec": time,

        "turning_angle_3d_deg": turn_3d,

        "turning_angle_xy_deg": turn_xy_raw,
        "turning_angle_yz_deg": turn_yz_raw,

        "turning_angle_xy_unwrapped_deg": turn_xy,
        "turning_angle_yz_unwrapped_deg": turn_yz,

        "turning_velocity_deg_s": turn_3d / DT,
    })

    out_path = os.path.join(OUTPUT_DIR, f"{trial}_TURNING_ANGLE.csv")
    out_df.to_csv(out_path, index=False)

    print(f"  → Saved: {out_path}")

    # ── SAVE TRIMMED (SAME FOLDER) ───────────────────────────────────
    if len(out_df) > 2 * TRIM:
        trimmed_df = out_df.iloc[TRIM:-TRIM].copy()

        trim_path = os.path.join(
            OUTPUT_DIR,
            f"{trial}_TURNING_ANGLE_TRIMMED.csv"
        )

        trimmed_df.to_csv(trim_path, index=False)

        print(f"  → Saved TRIMMED: {trim_path}")

    else:
        print("  ⚠ Too short to trim")

print("\n✅ DONE: Turning angles (250 fps) + trimmed saved in turning_angle folder")

Processing: Trial2_180rpmxyzpts_SUB.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based\Trial2_180rpmxyzpts_TURNING_ANGLE.csv
  → Saved TRIMMED: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based\Trial2_180rpmxyzpts_TURNING_ANGLE_TRIMMED.csv
Processing: Trial2_200rpmxyzpts_SUB.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based\Trial2_200rpmxyzpts_TURNING_ANGLE.csv
  → Saved TRIMMED: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based\Trial2_200rpmxyzpts_TURNING_ANGLE_TRIMMED.csv
Processing: Trial3_180rpmxyzpts_SUB.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based\Trial3_180rpmxyzpts_TURNING_ANGLE.csv
  → Saved TRIMMED: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based\Trial3_180rpmxyzpts_TURNING_ANGLE_TRIMMED.csv
Proc

In [7]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import Normalize
import plotly.graph_objects as go

# ── Config ─────────────────────────────────────────────────────────────
TURN_DIR = r"./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based"
SUB_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled"

FIG_DIR  = r"./../../dataFolders/MuscaChasingBeads/figures/TurningAnglePlots/subsample_turning_angle_center-based"
HTML_OUT = r"./../../dataFolders/MuscaChasingBeads/figures/TurningAnglePlots/ALL_TRIALS_3D.html"

os.makedirs(FIG_DIR, exist_ok=True)

# ── Get unique trials (avoid duplicates) ───────────────────────────────
all_files = glob.glob(os.path.join(TURN_DIR, "*_TURNING_ANGLE.csv"))

trial_map = {}
for f in all_files:
    fname = os.path.basename(f)
    trial = fname.split("_TURNING_ANGLE")[0]
    if trial not in trial_map:
        trial_map[trial] = f

turn_files = list(trial_map.values())

if not turn_files:
    raise FileNotFoundError("No turning angle files found")

# ── Plotly setup ───────────────────────────────────────────────────────
fig3d = go.Figure()
buttons = []
valid_trials = []

# ── MAIN LOOP ──────────────────────────────────────────────────────────
for path in turn_files:
    fname = os.path.basename(path)
    trial = fname.replace("_TURNING_ANGLE.csv", "")

    print(f"Processing: {trial}")

    df = pd.read_csv(path)
    time = df["time_sec"].values

    # ── 2D PLOTS ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    fig.suptitle(trial, fontsize=12)

    for ax, col, label, color in zip(
        axes,
        ["turning_angle_3d_deg", "turning_angle_xy_deg", "turning_angle_yz_deg"],
        ["3D turning angle",     "XY turning angle",     "YZ turning angle"],
        ["steelblue",            "darkorange",            "seagreen"]
    ):
        ax.plot(time, df[col].values, lw=0.8, color=color)
        ax.set_ylabel("degrees")
        ax.set_title(label)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("time (s)")
    plt.tight_layout()

    fig.savefig(os.path.join(FIG_DIR, f"{trial}_turning_angle.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

    print("  → Saved 2D plot")

    # ── 3D DATA ────────────────────────────────────────────────────────
    sub_path = glob.glob(os.path.join(SUB_DIR, f"{trial}*_SUB.csv"))
    if not sub_path:
        print(f"  ⚠ No SUB file for {trial}")
        continue

    df_sub = pd.read_csv(sub_path[0])

    head = df_sub[["pt2_X", "pt2_Y", "pt2_Z"]].values
    turn = df["turning_angle_3d_deg"].values

    min_len = min(len(head), len(turn))
    head = head[:min_len]
    turn = turn[:min_len]

    x, y, z = head[:, 0], head[:, 1], head[:, 2]

    # ── CLEAN NaNs / Infs ──────────────────────────────────────────────
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & np.isfinite(turn)
    x, y, z, turn = x[valid], y[valid], z[valid], turn[valid]

    if len(x) < 2:
        print(f"  ⚠ Skipping {trial} (insufficient valid data)")
        continue

    # ── STATIC 3D (COLORED LINE LIKE PLOTLY) ───────────────────────────
    fig_static = plt.figure(figsize=(7, 6))
    ax3d = fig_static.add_subplot(111, projection='3d')

    norm = Normalize(vmin=np.min(turn), vmax=np.max(turn))
    cmap = cm.get_cmap('turbo')

    # Draw colored line segments
    for i in range(len(x) - 1):
        color = cmap(norm(turn[i]))
        ax3d.plot(
            x[i:i+2],
            y[i:i+2],
            z[i:i+2],
            color=color,
            linewidth=2
        )

    # Colorbar
    mappable = cm.ScalarMappable(norm=norm, cmap=cmap)
    mappable.set_array(turn)
    cbar = fig_static.colorbar(mappable, ax=ax3d, pad=0.1)
    cbar.set_label("Turning Angle (°)")

    ax3d.set_title(f"{trial} — 3D Trajectory (Turning Angle)", fontsize=10)
    ax3d.set_xlabel("X")
    ax3d.set_ylabel("Y")
    ax3d.set_zlabel("Z")

    # Equal scaling
    max_range = np.array([
        x.max()-x.min(),
        y.max()-y.min(),
        z.max()-z.min()
    ]).max() / 2.0

    if max_range == 0:
        max_range = 1e-6

    mid_x = (x.max()+x.min()) * 0.5
    mid_y = (y.max()+y.min()) * 0.5
    mid_z = (z.max()+z.min()) * 0.5

    ax3d.set_xlim(mid_x - max_range, mid_x + max_range)
    ax3d.set_ylim(mid_y - max_range, mid_y + max_range)
    ax3d.set_zlim(mid_z - max_range, mid_z + max_range)

    ax3d.view_init(elev=20, azim=135)

    fig_static.savefig(
        os.path.join(FIG_DIR, f"{trial}_3D_turning.png"),
        dpi=200,
        bbox_inches="tight"
    )
    plt.close(fig_static)

    print("  → Saved static 3D plot")

    # ── INTERACTIVE 3D (COMBINED HTML) ─────────────────────────────────
    visible = True if len(valid_trials) == 0 else False

    fig3d.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines+markers",
        visible=visible,
        marker=dict(size=3, color=turn, colorscale="Turbo"),
        line=dict(color=turn, colorscale="Turbo", width=3),
        name=trial
    ))

    valid_trials.append(trial)

# ── Dropdown ───────────────────────────────────────────────────────────
n_traces = len(valid_trials)

buttons = []
for i, trial in enumerate(valid_trials):
    visibility = [False] * n_traces
    visibility[i] = True

    buttons.append(dict(
        label=trial,
        method="update",
        args=[{"visible": visibility},
              {"title": f"{trial} — 3D Trajectory"}]
    ))

fig3d.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0.02,
        y=1.05
    )],
    title="3D Trajectory Viewer",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=900,
    height=700,
    scene_camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
)

fig3d.write_html(HTML_OUT, include_plotlyjs=True, full_html=True)

print("\n DONE")
print(f"→ 2D + static 3D plots: {FIG_DIR}")
print(f"→ Interactive HTML: {HTML_OUT}")

Processing: Trial2_180rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_24880\3948204650.py:104: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial2_200rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_24880\3948204650.py:104: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial3_180rpmxyzpts
  → Saved 2D plot


...

  → Saved static 3D plot
Processing: Trial5_400rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_24880\3948204650.py:104: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial7_400rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_24880\3948204650.py:104: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot

✅ DONE
→ 2D + static 3D plots: ./../../dataFolders/MuscaChasingBeads/figures/TurningAnglePlots/subsample_turning_angle_center-based
→ Interactive HTML: ./../../dataFolders/MuscaChasingBeads/figures/TurningAnglePlots/ALL_TRIALS_3D.html


In [8]:
# removing edge effect
import os
import glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ── PATHS ─────────────────────────────────────────────────────────────
TURN_DIR = r"./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based"
SUB_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled"

OUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based"
os.makedirs(OUT_DIR, exist_ok=True)

# ── GET UNIQUE TRIAL FILES ────────────────────────────────────────────
all_files = glob.glob(os.path.join(TURN_DIR, "*_TURNING_ANGLE.csv"))

trial_map = {}
for f in all_files:
    fname = os.path.basename(f)
    trial = fname.split("_TURNING_ANGLE")[0]
    if trial not in trial_map:
        trial_map[trial] = f

turn_files = list(trial_map.values())

if not turn_files:
    raise FileNotFoundError("No turning angle files found")

# ── MAIN LOOP ─────────────────────────────────────────────────────────
for path in turn_files:
    fname = os.path.basename(path)
    trial = fname.replace("_TURNING_ANGLE.csv", "")

    print(f"Processing: {trial}")

    df = pd.read_csv(path)

    # ── MATCH SUBSAMPLED TRAJECTORY ───────────────────────────────────
    sub_path = glob.glob(os.path.join(SUB_DIR, f"{trial}*_SUB.csv"))
    if not sub_path:
        print(f" No SUB file found")
        continue

    df_sub = pd.read_csv(sub_path[0])

    head = df_sub[["pt2_X", "pt2_Y", "pt2_Z"]].values
    turn = df["turning_angle_3d_deg"].values

    # ── ALIGN LENGTHS ─────────────────────────────────────────────────
    min_len = min(len(head), len(turn))
    head = head[:min_len]
    turn = turn[:min_len]

    x, y, z = head[:, 0], head[:, 1], head[:, 2]

    # ── CLEAN NaNs / Infs ─────────────────────────────────────────────
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & np.isfinite(turn)
    x, y, z, turn = x[valid], y[valid], z[valid], turn[valid]

    if len(x) < 10:
        print(" Too few valid points")
        continue

    # ── REMOVE EDGE EFFECT (KEY STEP) ─────────────────────────────────
    x = x[4:-4]
    y = y[4:-4]
    z = z[4:-4]
    turn = turn[4:-4]

    if len(x) < 2:
        print("  Too short after trimming")
        continue

    # ── CREATE INTERACTIVE 3D PLOT ────────────────────────────────────
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="lines+markers",
        marker=dict(
            size=3,
            color=turn,
            colorscale="Turbo",
            colorbar=dict(title="Turning Angle (°)")
        ),
        line=dict(
            color=turn,
            colorscale="Turbo",
            width=4
        ),
        name=trial
    ))

    fig.update_layout(
        title=f"{trial} — 3D Trajectory (Edge-trimmed)",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z"
        ),
        width=900,
        height=700
    )

    # ── SAVE ─────────────────────────────────────────────────────────
    out_path = os.path.join(OUT_DIR, f"{trial}_3D_cleaned.html")

    fig.write_html(out_path, include_plotlyjs=True, full_html=True)

    print(f"  → Saved: {out_path}")

print("\n DONE: Cleaned interactive 3D plots generated")

Processing: Trial2_180rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based\Trial2_180rpmxyzpts_3D_cleaned.html
Processing: Trial2_200rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based\Trial2_200rpmxyzpts_3D_cleaned.html
Processing: Trial3_180rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based\Trial3_180rpmxyzpts_3D_cleaned.html
Processing: Trial4_400rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based\Trial4_400rpmxyzpts_3D_cleaned.html
Processing: Trial5_180rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based\Trial5_180rpmxyzpts_3D_cleaned.html
Processing: Trial5_400rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAngl

In [9]:
# removing edge effect (2d plots)
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── PATHS ─────────────────────────────────────────────────────────────
TURN_DIR = r"./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based"

OUT_2D_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based/plots_2D_cleaned"
os.makedirs(OUT_2D_DIR, exist_ok=True)

# ── GET UNIQUE FILES ──────────────────────────────────────────────────
all_files = glob.glob(os.path.join(TURN_DIR, "*_TURNING_ANGLE.csv"))

trial_map = {}
for f in all_files:
    fname = os.path.basename(f)
    trial = fname.split("_TURNING_ANGLE")[0]
    if trial not in trial_map:
        trial_map[trial] = f

turn_files = list(trial_map.values())

if not turn_files:
    raise FileNotFoundError("No turning angle files found")

# ── MAIN LOOP ─────────────────────────────────────────────────────────
for path in turn_files:
    fname = os.path.basename(path)
    trial = fname.replace("_TURNING_ANGLE.csv", "")

    print(f"\nProcessing: {trial}")

    df = pd.read_csv(path)

    turn_3d = df["turning_angle_3d_deg"].values
    turn_xy = df["turning_angle_xy_deg"].values
    turn_yz = df["turning_angle_yz_deg"].values
    time    = df["time_sec"].values

    # ── CLEAN NaNs / Infs ─────────────────────────────────────────────
    valid = (
        np.isfinite(turn_3d) &
        np.isfinite(turn_xy) &
        np.isfinite(turn_yz) &
        np.isfinite(time)
    )

    turn_3d = turn_3d[valid]
    turn_xy = turn_xy[valid]
    turn_yz = turn_yz[valid]
    time    = time[valid]

    if len(time) < 10:
        print("  ⚠ Too few valid points")
        continue

    # ── REMOVE EDGE EFFECT ────────────────────────────────────────────
    turn_3d = turn_3d[4:-4]
    turn_xy = turn_xy[4:-4]
    turn_yz = turn_yz[4:-4]
    time    = time[4:-4]

    if len(time) < 2:
        print("  ⚠ Too short after trimming")
        continue

    # ── 📈 2D PLOTS ──────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

    axes[0].plot(time, turn_3d, lw=1)
    axes[0].set_title("3D Turning Angle")
    axes[0].set_ylabel("deg")

    axes[1].plot(time, turn_xy, lw=1)
    axes[1].set_title("XY Turning Angle")
    axes[1].set_ylabel("deg")

    axes[2].plot(time, turn_yz, lw=1)
    axes[2].set_title("YZ Turning Angle")
    axes[2].set_ylabel("deg")
    axes[2].set_xlabel("time (s)")

    for ax in axes:
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"{trial} — Edge-trimmed Turning Angles")
    fig.tight_layout()

    # ── SAVE ─────────────────────────────────────────────────────────
    out_path = os.path.join(OUT_2D_DIR, f"{trial}_turning_2D_cleaned.png")
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

    print(f"  → Saved: {out_path}")

print("\n DONE: Cleaned 2D plots generated")


Processing: Trial2_180rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based/plots_2D_cleaned\Trial2_180rpmxyzpts_turning_2D_cleaned.png

Processing: Trial2_200rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based/plots_2D_cleaned\Trial2_200rpmxyzpts_turning_2D_cleaned.png

Processing: Trial3_180rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based/plots_2D_cleaned\Trial3_180rpmxyzpts_turning_2D_cleaned.png

Processing: Trial4_400rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based/plots_2D_cleaned\Trial4_400rpmxyzpts_turning_2D_cleaned.png

Processing: Trial5_180rpmxyzpts
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based/plots_2D_cleaned\Trial5_180rpmxyzpt

# error angle

In [4]:
import os
import glob
import numpy as np
import pandas as pd

# ── Config ─────────────────────────────────────────────────────────────
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────
def compute_error_angle_head(bead, head, center):
    v_heading = head - center
    v_bead    = bead - head

    dot   = np.sum(v_heading * v_bead, axis=1)
    cross = np.linalg.norm(np.cross(v_heading, v_bead), axis=1)

    return np.degrees(np.arctan2(cross, dot))  # 0–180 (no unwrap)

def compute_hori_error(bead, head, center):
    v_heading = (head - center)[:, :2]
    v_bead    = (bead - head)[:, :2]

    dot   = np.sum(v_heading * v_bead, axis=1)
    cross = v_heading[:,0]*v_bead[:,1] - v_heading[:,1]*v_bead[:,0]

    return np.degrees(np.arctan2(cross, dot))  # -180 → 180

def compute_ver_error(bead, head, center):
    v_heading = (head - center)[:, 1:3]
    v_bead    = (bead - head)[:, 1:3]

    dot   = np.sum(v_heading * v_bead, axis=1)
    cross = v_heading[:,0]*v_bead[:,1] - v_heading[:,1]*v_bead[:,0]

    return np.degrees(np.arctan2(cross, dot))  # -180 → 180

# ── Process ───────────────────────────────────────────────────────────
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SUB.csv"))

for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace("_SUB.csv", "")
    print(f"Processing: {fname}")

    df     = pd.read_csv(path)
    frames = df["frame"].values

    bead   = df[["pt1_X","pt1_Y","pt1_Z"]].values
    head   = df[["pt2_X","pt2_Y","pt2_Z"]].values
    center = df[["center_X","center_Y","center_Z"]].values

    # ── Distance ─────────────────────────────────────────────────────
    dist_head = np.linalg.norm(bead - head, axis=1)

    # ── Error angles ─────────────────────────────────────────────────
    err_head = compute_error_angle_head(bead, head, center)

    err_hori_raw = compute_hori_error(bead, head, center)
    err_vert_raw = compute_ver_error(bead, head, center)

    # ── UNWRAP (ONLY FOR SIGNED ANGLES) ──────────────────────────────
    err_hori_unwrapped = np.degrees(np.unwrap(np.radians(err_hori_raw)))
    err_vert_unwrapped = np.degrees(np.unwrap(np.radians(err_vert_raw)))

    # ── Output ───────────────────────────────────────────────────────
    out_df = pd.DataFrame({
        "frame": frames,

        # distance
        "dist_head_m": dist_head,

        # 3D error (magnitude only)
        "error_angle_head_deg": err_head,

        # horizontal (wrapped + unwrapped)
        "error_angle_horiz_deg": err_hori_raw,
        "error_angle_horiz_unwrapped_deg": err_hori_unwrapped,

        # vertical (wrapped + unwrapped)
        "error_angle_vert_deg": err_vert_raw,
        "error_angle_vert_unwrapped_deg": err_vert_unwrapped,
    })

    out_path = os.path.join(OUTPUT_DIR, f"{trial}_CHASE_METRICS.csv")
    out_df.to_csv(out_path, index=False)

    print(f"  → Saved: {out_path}")

print("\n✅ DONE: Error angles computed with proper unwrapping")

Processing: Trial2_180rpmxyzpts_SUB.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled\Trial2_180rpmxyzpts_CHASE_METRICS.csv
Processing: Trial2_200rpmxyzpts_SUB.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled\Trial2_200rpmxyzpts_CHASE_METRICS.csv
Processing: Trial3_180rpmxyzpts_SUB.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled\Trial3_180rpmxyzpts_CHASE_METRICS.csv
Processing: Trial4_400rpmxyzpts_SUB.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled\Trial4_400rpmxyzpts_CHASE_METRICS.csv
Processing: Trial5_180rpmxyzpts_SUB.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled\Trial5_180rpmxyzpts_CHASE_METRICS.csv
Processing: Trial5_400rpmxyzpts_SUB.csv
  → Saved: ./../../dataFolders

In [5]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import Normalize
import plotly.graph_objects as go

# ── Config ─────────────────────────────────────────────────────────────
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled"
SUB_DIR    = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled"

FIG_DIR    = r"./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled"
HTML_OUT   = r"./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled/ALL_TRIALS_3D.html"

os.makedirs(FIG_DIR, exist_ok=True)

FPS = 250

# ── Get unique trials (avoid duplicates) ───────────────────────────────
all_files = glob.glob(os.path.join(INPUT_DIR, "*_CHASE_METRICS.csv"))

trial_map = {}
for f in all_files:
    fname = os.path.basename(f)
    trial = fname.split("_CHASE_METRICS")[0]
    if trial not in trial_map:
        trial_map[trial] = f

csv_files = list(trial_map.values())

if not csv_files:
    raise FileNotFoundError("No CHASE_METRICS CSV files found")

# ── Plotly setup ───────────────────────────────────────────────────────
fig3d = go.Figure()
buttons = []
valid_trials = []

# ── MAIN LOOP ──────────────────────────────────────────────────────────
for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace("_CHASE_METRICS.csv", "")

    print(f"Processing: {trial}")

    df = pd.read_csv(path)
    time = df["frame"].values / FPS

    # ── 2D PLOTS ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(trial, fontsize=12)

    for ax, col, label, color in zip(
        axes,
        ["error_angle_head_deg", "error_angle_horiz_deg", "error_angle_vert_deg", "dist_head_m"],
        ["3D error angle",       "Horizontal error (XY)", "Vertical error (YZ)",  "Distance head→bead"],
        ["steelblue",            "darkorange",             "seagreen",             "mediumpurple"]
    ):
        ax.plot(time, df[col].values, lw=0.8, color=color)
        ax.set_ylabel("deg" if "deg" in col else "m")
        ax.set_title(label)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("time (s)")
    plt.tight_layout()

    fig.savefig(os.path.join(FIG_DIR, f"{trial}_error_angle.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

    print("  → Saved 2D plot")

    # ── 3D DATA ────────────────────────────────────────────────────────
    sub_path = glob.glob(os.path.join(SUB_DIR, f"{trial}*_SUB.csv"))
    if not sub_path:
        print(f"  ⚠ No SUB file for {trial}")
        continue

    df_sub = pd.read_csv(sub_path[0])

    head = df_sub[["pt2_X", "pt2_Y", "pt2_Z"]].values
    err  = df["error_angle_head_deg"].values

    min_len = min(len(head), len(err))
    head = head[:min_len]
    err  = err[:min_len]

    x, y, z = head[:, 0], head[:, 1], head[:, 2]

    # ── CLEAN NaNs / Infs ──────────────────────────────────────────────
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & np.isfinite(err)
    x, y, z, err = x[valid], y[valid], z[valid], err[valid]

    if len(x) < 2:
        print(f"  ⚠ Skipping {trial} (insufficient valid data)")
        continue

    # ── STATIC 3D (COLORED LINE) ───────────────────────────────────────
    fig_static = plt.figure(figsize=(7, 6))
    ax3d = fig_static.add_subplot(111, projection='3d')

    norm = Normalize(vmin=np.min(err), vmax=np.max(err))
    cmap = cm.get_cmap('turbo')

    # Draw colored trajectory segments
    for i in range(len(x) - 1):
        color = cmap(norm(err[i]))
        ax3d.plot(
            x[i:i+2],
            y[i:i+2],
            z[i:i+2],
            color=color,
            linewidth=2
        )

    # Colorbar
    mappable = cm.ScalarMappable(norm=norm, cmap=cmap)
    mappable.set_array(err)
    cbar = fig_static.colorbar(mappable, ax=ax3d, pad=0.1)
    cbar.set_label("Error Angle (°)")

    ax3d.set_title(f"{trial} — 3D Trajectory (Error Angle)", fontsize=10)
    ax3d.set_xlabel("X")
    ax3d.set_ylabel("Y")
    ax3d.set_zlabel("Z")

    # Equal scaling
    max_range = np.array([
        x.max()-x.min(),
        y.max()-y.min(),
        z.max()-z.min()
    ]).max() / 2.0

    if max_range == 0:
        max_range = 1e-6

    mid_x = (x.max()+x.min()) * 0.5
    mid_y = (y.max()+y.min()) * 0.5
    mid_z = (z.max()+z.min()) * 0.5

    ax3d.set_xlim(mid_x - max_range, mid_x + max_range)
    ax3d.set_ylim(mid_y - max_range, mid_y + max_range)
    ax3d.set_zlim(mid_z - max_range, mid_z + max_range)

    ax3d.view_init(elev=20, azim=135)

    fig_static.savefig(
        os.path.join(FIG_DIR, f"{trial}_3D_error.png"),
        dpi=200,
        bbox_inches="tight"
    )
    plt.close(fig_static)

    print("  → Saved static 3D plot")

    # ── INTERACTIVE 3D (COMBINED HTML) ─────────────────────────────────
    visible = True if len(valid_trials) == 0 else False

    fig3d.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines+markers",
        visible=visible,
        marker=dict(size=3, color=err, colorscale="Turbo"),
        line=dict(color=err, colorscale="Turbo", width=3),
        name=trial
    ))

    valid_trials.append(trial)

# ── Dropdown ───────────────────────────────────────────────────────────
n_traces = len(valid_trials)

for i, trial in enumerate(valid_trials):
    visibility = [False] * n_traces
    visibility[i] = True

    buttons.append(dict(
        label=trial,
        method="update",
        args=[{"visible": visibility},
              {"title": f"{trial} — 3D Trajectory (Error Angle)"}]
    ))

fig3d.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0.02,
        y=1.05
    )],
    title="3D Error Angle Trajectory Viewer",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=900,
    height=700,
    scene_camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
)

fig3d.write_html(HTML_OUT, include_plotlyjs=True, full_html=True)

print("\n DONE")
print(f"→ 2D plots: {FIG_DIR}")
print(f"→ Static 3D plots: {FIG_DIR}")
print(f"→ Interactive HTML: {HTML_OUT}")

Processing: Trial2_180rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_24880\1955002215.py:106: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial2_200rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_24880\1955002215.py:106: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial3_180rpmxyzpts
  → Saved 2D plot


...

  → Saved static 3D plot
Processing: Trial5_400rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_24880\1955002215.py:106: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial7_400rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_24880\1955002215.py:106: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot

✅ DONE
→ 2D plots: ./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled
→ Static 3D plots: ./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled
→ Interactive HTML: ./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled/ALL_TRIALS_3D.html


# Speed 

In [11]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import decimate

# ── PATHS ─────────────────────────────────────────────────────────────
INPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Speed"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_250fps"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS_ORIG = 2500
FPS_NEW  = 250

# ── LOOP ─────────────────────────────────────────────────────────────
csv_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))

for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace(".csv", "")

    print(f"\nProcessing (250 fps): {trial}")

    df = pd.read_csv(path)

    if "speed" not in df.columns:
        print("  ⚠ No speed column")
        continue

    speed = df["speed"].values

    # ── DOWNSAMPLE ───────────────────────────────────────────────────
    speed_250 = decimate(speed, 10, ftype='fir', zero_phase=True)

    # ── SAVE ─────────────────────────────────────────────────────────
    out_df = pd.DataFrame({
        "frame": np.arange(len(speed_250)),
        "speed": speed_250
    })

    out_path = os.path.join(OUTPUT_DIR, f"{trial}_SPEED_250fps.csv")
    out_df.to_csv(out_path, index=False)

    print(f"  → Saved: {out_path}")

    # ── TIME ─────────────────────────────────────────────────────────
    t_orig = np.arange(len(speed)) / FPS_ORIG
    t_250  = np.arange(len(speed_250)) / FPS_NEW


print("\n DONE: 250 fps speed")


Processing (250 fps): Trial2_180rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_250fps\Trial2_180rpmxyzpts_SPEED_SPEED_250fps.csv

Processing (250 fps): Trial2_200rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_250fps\Trial2_200rpmxyzpts_SPEED_SPEED_250fps.csv

Processing (250 fps): Trial3_180rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_250fps\Trial3_180rpmxyzpts_SPEED_SPEED_250fps.csv

Processing (250 fps): Trial4_400rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_250fps\Trial4_400rpmxyzpts_SPEED_SPEED_250fps.csv

Processing (250 fps): Trial5_180rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_250fps\Trial5_180rpmxyzpts_SPEED_SPEED_250fps.csv

Processing (250 fps): Trial5_400rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_250fps\Trial5_400rpmxyzp

In [13]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── PATHS ─────────────────────────────────────────────────────────────
INPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_250fps"

OUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/Speed/250fps"
os.makedirs(OUT_DIR, exist_ok=True)

FPS = 250

# ── LOOP ─────────────────────────────────────────────────────────────
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_250fps.csv"))

if not csv_files:
    raise FileNotFoundError("No 250 fps speed files found")

for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace("_SPEED_250fps.csv", "")

    print(f"Processing: {trial}")

    df = pd.read_csv(path)

    if "speed" not in df.columns:
        print("  ⚠ No speed column")
        continue

    speed = df["speed"].values

    # ── TIME AXIS ────────────────────────────────────────────────────
    time = np.arange(len(speed)) / FPS

    # ── CLEAN NaNs ───────────────────────────────────────────────────
    valid = np.isfinite(speed)
    speed = speed[valid]
    time  = time[valid]

    if len(speed) < 10:
        print("  ⚠ Too few valid points")
        continue

    # ── PLOT ─────────────────────────────────────────────────────────
    plt.figure(figsize=(10,5))

    plt.plot(time, speed, lw=1.5)

    plt.title(f"{trial} — Speed (250 fps)")
    plt.xlabel("Time (s)")
    plt.ylabel("Speed")
    plt.grid(True)

    plt.tight_layout()

    # ── SAVE ─────────────────────────────────────────────────────────
    out_path = os.path.join(OUT_DIR, f"{trial}_speed_250fps.png")
    plt.savefig(out_path, dpi=150)
    plt.close()

    print(f"  → Saved: {out_path}")

print("\n DONE: All 250 fps speed plots saved")

Processing: Trial2_180rpmxyzpts_SPEED
  ⚠ Too few valid points
Processing: Trial2_200rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Speed/250fps\Trial2_200rpmxyzpts_SPEED_speed_250fps.png
Processing: Trial3_180rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Speed/250fps\Trial3_180rpmxyzpts_SPEED_speed_250fps.png
Processing: Trial4_400rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Speed/250fps\Trial4_400rpmxyzpts_SPEED_speed_250fps.png
Processing: Trial5_180rpmxyzpts_SPEED
  ⚠ Too few valid points
Processing: Trial5_400rpmxyzpts_SPEED
  ⚠ Too few valid points
Processing: Trial7_400rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Speed/250fps\Trial7_400rpmxyzpts_SPEED_speed_250fps.png

✅ DONE: All 250 fps speed plots saved
